<a href="https://colab.research.google.com/github/LaiTechTinker/VAE-Variational-autoencoder-implementation-/blob/main/VAE_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook i will be implenting Variational autoencoder in pytorch.The following are the task outline for me to do:
1.import libraries
2.Load Dataset
3.Define VAE architecture
4.Define the loss function and Optimizer
5.inititate training loop Block
6.Visualize the Training
7.Testing/image generation


In [ ]:
# !pip install tensorboard

In [ ]:
#Task 1 importing libraries
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torchvision
import torchvision.utils as vutils
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import save_image

In [ ]:
#Task 2 :Loading of Dataset
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_dataset=datasets.CIFAR10(root="./data",train=True,download=True,transform=transform)
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)

In [ ]:
example,_=next(iter(train_loader))
print(example[0].shape)#i'm checking for the channel and also shape of the image here
print(example[0])

torch.Size([3, 32, 32])
tensor([[[-0.4667, -0.4510, -0.4588,  ...,  0.0275, -0.0275, -0.0588],
         [-0.0745, -0.0902, -0.0824,  ...,  0.0510,  0.0275,  0.0118],
         [-0.1216, -0.1059, -0.0902,  ...,  0.0039,  0.0039,  0.0275],
         ...,
         [-0.4275, -0.0431,  0.4039,  ..., -0.2078, -0.2706, -0.1922],
         [-0.3882, -0.3412,  0.0745,  ..., -0.0980, -0.1373, -0.1137],
         [-0.3020, -0.3882, -0.3176,  ..., -0.0745, -0.0353,  0.0039]],

        [[-0.3804, -0.3569, -0.3647,  ..., -0.1294, -0.1686, -0.1922],
         [ 0.0039, -0.0039, -0.0039,  ..., -0.1529, -0.1686, -0.1686],
         [-0.0039,  0.0118,  0.0275,  ..., -0.2784, -0.2706, -0.2392],
         ...,
         [-0.4275, -0.0980,  0.3490,  ..., -0.1843, -0.2078, -0.1608],
         [-0.3961, -0.3725,  0.0510,  ..., -0.1216, -0.1529, -0.1608],
         [-0.3176, -0.3725, -0.3020,  ..., -0.1216, -0.1137, -0.0745]],

        [[-0.5373, -0.5216, -0.5294,  ..., -0.0667, -0.1059, -0.1294],
         [-0.3647, -0

In [ ]:
import torch
import torch.nn as nn


class VAE(nn.Module):

    def __init__(self, latent_dim=128):

        super().__init__()

        self.latent_dim = latent_dim

        # -------------------------
        # Encoder
        # -------------------------
        self.encoder = nn.Sequential()

        self.encoder.add_module(
            "conv1",
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1)
        )  # -> 32 x 16 x 16
        self.encoder.add_module("relu1", nn.ReLU())

        self.encoder.add_module(
            "conv2",
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        )  # -> 64 x 8 x 8
        self.encoder.add_module("relu2", nn.ReLU())

        self.encoder.add_module(
            "conv3",
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        )  # -> 128 x 4 x 4
        self.encoder.add_module("relu3", nn.ReLU())


        # -------------------------
        # Latent Space
        # -------------------------
        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)


        # -------------------------
        # Decoder Input
        # -------------------------
        self.decoder_input = nn.Linear(latent_dim, 128 * 4 * 4)


        # -------------------------
        # Decoder
        # -------------------------
        self.decoder = nn.Sequential()

        self.decoder.add_module(
            "trans1",
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        )  # -> 64 x 8 x 8
        self.decoder.add_module("relu1", nn.ReLU())

        self.decoder.add_module(
            "trans2",
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        )  # -> 32 x 16 x 16
        self.decoder.add_module("relu2", nn.ReLU())

        self.decoder.add_module(
            "trans3",
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
        )  # -> 3 x 32 x 32
        self.decoder.add_module("tanh", nn.Tanh())


    def forward(self, x):

        # Encode image
        x = self.encoder(x)

        # Flatten
        x = torch.flatten(x, start_dim=1)

        # Get latent parameters
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        # Reparameterization trick
        z = self.reparameterize(mu, logvar)

        # Map latent → feature map
        x = self.decoder_input(z)

        # Reshape
        x = x.view(-1, 128, 4, 4)

        # Reconstruct image
        reconstruction = self.decoder(x)

        return reconstruction, mu, logvar


    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        z = mu + eps * std

        return z

In [ ]:
def vae_loss(reconx, x, mu, logvar):

    # Reconstruction loss
    recon_loss = nn.functional.mse_loss(reconx, x, reduction="mean")

    # KL divergence
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + kl_loss

In [ ]:
writer = SummaryWriter("runs/vae_training")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 50
lr = 1e-3

model = VAE().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):

    total_loss = 0

    for batch_idx,(images, _) in enumerate(train_loader):

        images = images.to(device)

        optimizer.zero_grad()

        recon_images, mu, logvar = model(images)

        loss = vae_loss(recon_images, images, mu, logvar)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
           # Log only first batch to TensorBoard
        if batch_idx == 0:

            comparison = torch.cat([images[:8], recon_images[:8]])

            grid = torchvision.utils.make_grid(comparison, nrow=8)

            writer.add_image(
                "Original image (top) vs Reconstructed image (bottom)",
                grid,
                epoch
            )

    avg_loss = total_loss / len(train_loader)

    writer.add_scalar("Loss/train", avg_loss, epoch)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(train_loader):.4f}")

Epoch [1/50] Loss: 0.1904
Epoch [2/50] Loss: 0.1686
Epoch [3/50] Loss: 0.1655
Epoch [4/50] Loss: 0.1648
Epoch [5/50] Loss: 0.1642
Epoch [6/50] Loss: 0.1639
Epoch [7/50] Loss: 0.1637
Epoch [8/50] Loss: 0.1634
Epoch [9/50] Loss: 0.1628
Epoch [10/50] Loss: 0.1622
Epoch [11/50] Loss: 0.1621
Epoch [12/50] Loss: 0.1620
Epoch [13/50] Loss: 0.1617
Epoch [14/50] Loss: 0.1615
Epoch [15/50] Loss: 0.1614
Epoch [16/50] Loss: 0.1614
Epoch [17/50] Loss: 0.1613
Epoch [18/50] Loss: 0.1612
Epoch [19/50] Loss: 0.1611
Epoch [20/50] Loss: 0.1610
Epoch [21/50] Loss: 0.1610
Epoch [22/50] Loss: 0.1610
Epoch [23/50] Loss: 0.1608
Epoch [24/50] Loss: 0.1607
Epoch [25/50] Loss: 0.1608
Epoch [26/50] Loss: 0.1607
Epoch [27/50] Loss: 0.1606
Epoch [28/50] Loss: 0.1607
Epoch [29/50] Loss: 0.1606
Epoch [30/50] Loss: 0.1606
Epoch [31/50] Loss: 0.1604
Epoch [32/50] Loss: 0.1605
Epoch [33/50] Loss: 0.1605
Epoch [34/50] Loss: 0.1605
Epoch [35/50] Loss: 0.1605
Epoch [36/50] Loss: 0.1605
Epoch [37/50] Loss: 0.1606
Epoch [38/

In [ ]:
# ==== Save Trained Model ====
torch.save(model.state_dict(), './VAE.pth')

In [ ]:
import tensorboard

In [ ]:
!tensorboard --logdir=runs

2026-03-08 17:23:33.249287: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772990613.283844   42873 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772990613.295066   42873 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772990613.320390   42873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772990613.320438   42873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772990613.320455   42873 computation_placer.cc:177] computation placer alr

In [ ]:
#let's create an image reconstruction function from random noise
model=VAE().to(device)
model.load_state_dict(torch.load('./VAE.pth',map_location=device))
model.eval()
# initiate random noise generation
z=torch.randn(64,128).to(device)

with torch.no_grad():
  samples=model.decoder_input(z)
  samples=samples.view(-1,128,4,4)
  samples=model.decoder(samples)
vutils.save_image(samples,"./generatedimage.png",nrow=8,normalize=True)